# RangeLab free browser-based ES/MES five-minute exporter

Run this notebook in [Google Colab](https://colab.research.google.com/) from a Chromebook. It downloads public Yahoo Finance futures candles for `ES=F` and `MES=F` separately, writes raw and regular-hours CSV files, creates quality receipts, and downloads one ZIP.

**Limits:** public Yahoo intraday data is vendor-reported, delayed and currently limited to recent intraday history. The yfinance documentation says intraday data cannot extend beyond the last 60 days. This is a free smoke-test source, not a long-window exchange-direct artifact, not broker fills, and not proof of an ORB edge. Never substitute ES rows for MES rows.

In [ ]:
!pip -q install yfinance

In [ ]:
import hashlib
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

# Public Yahoo symbols are deliberately kept independent.
ASSETS = {"ES": "ES=F", "MES": "MES=F"}
INTERVAL = "5m"
PERIOD = "59d"  # below the documented 60-day intraday boundary
RTH_ONLY = True  # creates files sized for the RangeLab dashboard import
OUTPUT_DIR = Path("/content/rangelab_export")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def flatten_columns(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [str(col[0]) for col in frame.columns]
    frame.columns = [str(col).strip().lower().replace(" ", "_") for col in frame.columns]
    return frame

def iso_timestamp(index):
    # Preserve an explicit timezone. If the provider ever returns a naive
    # index, UTC is an explicit assumption and is disclosed in the receipt.
    assumed_naive_utc = index.tz is None
    if index.tz is None:
        index = index.tz_localize("UTC")
    else:
        index = index.tz_convert("UTC")
    return index, assumed_naive_utc

def download_asset(asset, yahoo_symbol):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{yahoo_symbol.replace('=', '%3D')}?range={PERIOD}&interval={INTERVAL}"
    frame = yf.download(
        yahoo_symbol,
        period=PERIOD,
        interval=INTERVAL,
        auto_adjust=False,
        actions=False,
        prepost=True,
        progress=False,
        threads=False,
    )
    if frame is None or frame.empty:
        raise RuntimeError(f"No rows returned for {asset} ({yahoo_symbol}).")
    frame = flatten_columns(frame)
    required = ["open", "high", "low", "close", "volume"]
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise RuntimeError(f"{asset}: missing Yahoo columns {missing}; no file written.")
    frame.index, naive_utc = iso_timestamp(frame.index)
    frame = frame[required].sort_index()
    duplicate_rows = int(frame.index.duplicated(keep=False).sum())
    frame = frame[~frame.index.duplicated(keep="first")]
    null_ohlc_rows = int(frame[["open", "high", "low", "close"]].isna().any(axis=1).sum())
    clean = frame.dropna(subset=["open", "high", "low", "close"]).copy()
    clean.index, _ = iso_timestamp(clean.index)

    et = clean.index.tz_convert("America/New_York")
    minutes = et.hour * 60 + et.minute
    rth_mask = (et.dayofweek < 5) & (minutes >= 9 * 60 + 30) & (minutes < 16 * 60)
    rth = clean.loc[rth_mask].copy()
    off_five_minute = int(((clean.index.asi8 // 1_000_000_000) % 300 != 0).sum())

    def to_export(frame_to_write):
        out = frame_to_write.reset_index().rename(columns={"Datetime": "timestamp", "index": "timestamp"})
        out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True).map(lambda value: value.isoformat())
        out["symbol"] = asset
        out["source_symbol"] = yahoo_symbol
        return out[["timestamp", "open", "high", "low", "close", "volume", "symbol", "source_symbol"]]

    raw_path = OUTPUT_DIR / f"{asset}-yahoo-{INTERVAL}-{PERIOD}.csv"
    rth_path = OUTPUT_DIR / f"{asset}-yahoo-{INTERVAL}-{PERIOD}-rth.csv"
    to_export(clean).to_csv(raw_path, index=False)
    to_export(rth).to_csv(rth_path, index=False)

    receipt = {
        "asset": asset,
        "source_symbol": yahoo_symbol,
        "provider": "Yahoo Finance chart endpoint via yfinance",
        "source_url": url,
        "acquired_at_utc": datetime.now(timezone.utc).isoformat(),
        "interval": INTERVAL,
        "requested_period": PERIOD,
        "raw_rows_after_duplicate_removal": int(len(clean)),
        "raw_rows_before_null_ohlc_removal": int(len(frame)),
        "rth_rows": int(len(rth)),
        "duplicate_rows_detected": duplicate_rows,
        "null_ohlc_rows_excluded_from_clean_files": null_ohlc_rows,
        "off_five_minute_rows": off_five_minute,
        "provider_index_was_naive_utc_assumption": naive_utc,
        "raw_csv_sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
        "rth_csv_sha256": hashlib.sha256(rth_path.read_bytes()).hexdigest(),
        "warning": "Vendor-reported recent intraday snapshot; no exchange-direct or broker-fill claim.",
    }
    receipt_path = OUTPUT_DIR / f"{asset}-receipt.json"
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")
    return receipt, raw_path, rth_path


In [ ]:
receipts = {}
files_to_zip = []
for asset, yahoo_symbol in ASSETS.items():
    receipt, raw_path, rth_path = download_asset(asset, yahoo_symbol)
    receipts[asset] = receipt
    files_to_zip.extend([raw_path, rth_path, OUTPUT_DIR / f"{asset}-receipt.json"])
    print(f"{asset}: {receipt['raw_rows_after_duplicate_removal']} clean rows; {receipt['rth_rows']} RTH rows")

zip_path = Path("/content/RangeLab_free_yahoo_5m_export.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, arcname=path.name)

print(f"Created {zip_path}")
print("Use the -rth.csv files for the dashboard import after reviewing the receipt files.")


In [ ]:
from google.colab import files
files.download(str(zip_path))

## Dashboard import

1. Unzip the download.
2. Open the `ES-...-rth.csv` or `MES-...-rth.csv` file.
3. In Range Lab, open **Market data & replay**, select the matching asset, and choose **Import**.
4. Import ES and MES separately.

The dashboard performs additional timestamp, regular-session, five-minute, OHLC, duplicate, gap and tick-grid checks. Keep the raw CSV and receipt privately; the CSV is not proof of a long-term edge.